# 06 — PLM Embeddings (Protein Language Models)

**Goal**: Protein language models (like ESM-2 or ProtBert) learn rich, contextual representations of amino acid sequences. We extract these embeddings and build predictive models over them.

**Verify gates**:
- Load a pre-trained PLM (`facebook/esm2_t33_650M_UR50D` for speed/memory efficiency)
- Extract and mean-pool the last hidden state for a few sample sequences
- Batch process all train & test sequences (save to `.npy`)
- Train Logistic Regression & a small MLP on the embeddings (Cluster CV)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')

from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, roc_curve, auc

plt.style.use('dark_background')
sns.set_palette('viridis')
print('Libraries loaded ✓')

## Step 6.1 — Load Model & Tokenizer

In [ ]:
MODEL_NAME = "facebook/esm2_t33_650M_UR50D" # Larger ESM-2 model (33 layers, 1280 dim)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME).to(device)
model.eval() # Set to evaluation mode

print(f'Loaded {MODEL_NAME} successfully.')

## Step 6.2 — Embedding Extraction Logic

In [ ]:
def get_embeddings(sequences, batch_size=32):
    """
    Extracts mean-pooled sequence embeddings from ESM-2.
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(sequences), batch_size), desc="Extracting PLM Embeddings"):
        batch_seqs = sequences[i : i + batch_size]
        
        # ESM tokenizer expects raw sequence strings
        inputs = tokenizer(batch_seqs, return_tensors="pt", padding=True, truncation=True, max_length=200)
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        with torch.no_grad():
            outputs = model(**inputs)
            # outputs.last_hidden_state has shape (batch, seq_len, hidden_dim)
            # We mask out pad tokens before mean pooling
            attention_mask = inputs['attention_mask'].unsqueeze(-1) # (batch, seq_len, 1)
            hidden_states = outputs.last_hidden_state * attention_mask
            
            # Sum along seq length and divide by actual length
            sum_embeddings = hidden_states.sum(dim=1)
            seq_lens = attention_mask.sum(dim=1)
            mean_embeddings = sum_embeddings / seq_lens
            
            all_embeddings.append(mean_embeddings.cpu().numpy())
            
    return np.vstack(all_embeddings)

# Quick Test
sample_seqs = ["MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG", "ACDEFGH"]
sample_emb = get_embeddings(sample_seqs, batch_size=2)
print(f'\nSample embedding shape: {sample_emb.shape}') # Should be (2, 320)

## Step 6.3 & 6.4 — Process All Data & Save

In [ ]:
train_df = pd.read_csv('../data/processed/train_clusters.csv')
test_df  = pd.read_csv('../data/processed/test_clean.csv')

# Compute embeddings
X_train_esm = get_embeddings(train_df['Sequence'].tolist(), batch_size=32)
X_test_esm  = get_embeddings(test_df['Sequence'].tolist(), batch_size=32)

print(f'\nTrain ESM shape: {X_train_esm.shape}')
print(f'Test ESM shape:  {X_test_esm.shape}')

np.save('../data/processed/X_train_esm.npy', X_train_esm)
np.save('../data/processed/X_test_esm.npy', X_test_esm)
print('Saved embeddings to .npy.')

## Step 6.5 — Evaluate Standalone PLM Representations (Cluster CV)

In [ ]:
y_train = train_df['Label'].values
cluster_labels = train_df['Cluster'].values

def evaluate_dense_cv(model_name, model_fn, scale=False):
    cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
    
    oof_preds = np.zeros(len(X_train_esm))
    fold_metrics = []
    tprs = []
    mean_fpr = np.linspace(0, 1, 100)
    
    fig, ax = plt.subplots(figsize=(8, 6))
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_esm, y_train, groups=cluster_labels)):
        X_tr, y_tr = X_train_esm[train_idx], y_train[train_idx]
        X_va, y_va = X_train_esm[val_idx], y_train[val_idx]
        
        if scale:
            scaler = StandardScaler()
            X_tr = scaler.fit_transform(X_tr)
            X_va = scaler.transform(X_va)
            
        model = model_fn()
        model.fit(X_tr, y_tr)
        
        preds = model.predict_proba(X_va)[:, 1] 
        oof_preds[val_idx] = preds
        
        auc_score = roc_auc_score(y_va, preds)
        fold_metrics.append(auc_score)
        
        fpr, tpr, _ = roc_curve(y_va, preds)
        interp_tpr = np.interp(mean_fpr, fpr, tpr)
        interp_tpr[0] = 0.0
        tprs.append(interp_tpr)
        
        ax.plot(fpr, tpr, lw=1, alpha=0.3, label=f'Fold {fold+1} (AUC = {auc_score:.3f})')

    mean_tpr = np.mean(tprs, axis=0)
    mean_tpr[-1] = 1.0
    mean_auc = auc(mean_fpr, mean_tpr)
    std_auc = np.std(fold_metrics)
    
    ax.plot(mean_fpr, mean_tpr, color='b', label=f'Mean ROC (AUC = {mean_auc:.3f} ± {std_auc:.3f})', lw=2)
    ax.plot([0, 1], [0, 1], linestyle='--', lw=2, color='r', alpha=0.8)
    
    ax.set_title(f'ROC Curve - {model_name} (ESM-2)', fontweight='bold')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.legend(loc="lower right")
    plt.show()
    
    print(f"{model_name} Cluster CV: Mean AUC = {mean_auc:.4f} ± {std_auc:.4f}")
    return mean_auc, oof_preds

print('Evaluating PLM Embeddings...')

# 1. Logistic Regression
lr_fn = lambda: LogisticRegression(max_iter=1000, random_state=42, C=0.1, n_jobs=-1)
lresm_auc, oof_lresm = evaluate_dense_cv("Logistic Regression", lr_fn, scale=True)

# 2. Multi-Layer Perceptron (since embeddings are highly non-linear)
mlp_fn = lambda: MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    max_iter=200,
    early_stopping=True,
    random_state=42
)
mlp_auc, oof_mlp = evaluate_dense_cv("MLP", mlp_fn, scale=True)

## Step 6.6 — Save ESM Out-of-Fold Predictions

In [ ]:
oof_df = pd.DataFrame({
    'Sequence': train_df['Sequence'],
    'Label': y_train,
    'esm_lr_pred': oof_lresm,
    'esm_mlp_pred': oof_mlp
})
oof_df.to_csv('../data/processed/oof_esm.csv', index=False)
print('Saved ESM out-of-fold predictions to data/processed/oof_esm.csv')
